In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 12. トークナイザー — テキスト テキスト ⭐

> **学習目標**
> - テキスト テキスト テキスト LLMテキスト テキスト テキスト
> - BPE テキスト テキスト Pythonテキスト テキスト
> - WordPiece, Unigram Language Modelテキスト テキスト テキスト
> - テキスト テキスト トークナイザー(Byte-level BPE)テキスト テキスト テキスト

## 12.1 テキスト テキスト

テキスト テキスト テキスト:
- **テキスト テキスト**: "Hello, world!" → ["Hello", ",", "world", "!"]
  - 問題: テキスト テキスト, テキスト/テキスト テキスト テキスト, OOV(Out-Of-Vocabulary)
- **テキスト テキスト**: "Hello" → ['H', 'e', 'l', 'l', 'o']
  - 問題: テキスト テキスト テキスト, テキスト テキスト テキスト
- **テキスト テキスト**: "unhappiness" → ["un", "happiness"] テキスト ["un", "happ", "iness"]
  - **LLM テキスト**: テキスト度 テキスト テキスト テキスト, テキスト テキスト テキスト

GPT-2: Byte-level BPE. BERT: WordPiece. T5: Unigram.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# テキスト テキスト テキスト テキスト
text = "The quick brown fox jumps over the lazy dog."
word_tokens = text.lower().replace('.', ' .').split()
print(f"word テキスト: {word_tokens}")

# Character-level
char_tokens = list(text.lower())
print(f"Character-level: {char_tokens}")

# テキスト (Conceptテキスト テキスト)
subword_tokens = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
print(f"テキスト テキスト (テキスト): {subword_tokens}")


## 12.2 Byte-Pair Encoding (BPE) — GPT-2テキスト テキスト

**テキスト**:
1. テキスト テキスト テキスト テキスト テキスト
2. テキスト テキスト度テキスト テキスト テキスト テキスト(2-gram)テキスト テキスト
3. テキスト テキスト 度テキスト テキスト テキスト

テキスト: 'low' (5テキスト), 'lower' (2テキスト), 'newest' (6テキスト), 'widest' (3テキスト)

テキスト度 テキスト 'e'+'s' テキスト 9テキスト → テキスト → 'es'


In [ ]:
# BPE テキスト テキスト
from collections import Counter

class BPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []  # (a, b) テキスト テキスト テキスト
        self.vocab = set()

    def _get_word_freqs(self, corpus):
        """word テキスト度 Computation."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for w in words:
                word_freqs[w] += 1
        return word_freqs

    def _get_pair_freqs(self, splits, word_freqs):
        """テキスト テキスト テキスト度 Computation."""
        pair_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
        return pair_freqs

    def _merge(self, pair, splits):
        """テキスト wordテキスト pairテキスト テキストSum."""
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = self._get_word_freqs(corpus)
        # テキスト wordテキスト Character-levelテキスト Decomposition (テキスト </w> Display)
        splits = {w: list(w) + ['</w>'] for w in word_freqs}
        # テキスト テキスト
        self.vocab = set(c for w in word_freqs for c in w) | {'</w>'}

        for _ in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(splits, word_freqs)
            if not pair_freqs:
                break
            best_pair = max(pair_freqs, key=pair_freqs.get)
            splits = self._merge(best_pair, splits)
            self.merges.append(best_pair)
            self.vocab.add(best_pair[0] + best_pair[1])

    def encode(self, word):
        """Trainingテキスト テキスト word エンコード."""
        split = list(word) + ['</w>']
        for a, b in self.merges:
            i = 0
            new_split = []
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            split = new_split
        return split

# テキスト テキスト Training
corpus = [
    "low low low low low",
    "lower lower newest newest newest",
    "widest widest newest newest",
]
bpe = BPE(num_merges=10)
bpe.train(corpus)

print("Trainingテキスト テキストSum テキスト:")
for i, (a, b) in enumerate(bpe.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")
print(f"\nVocabulary Size: {len(bpe.vocab)}")

# エンコード Test
for word in ['low', 'lowest', 'newer', 'widest']:
    tokens = bpe.encode(word)
    print(f"  {word:>10} -> {tokens}")


## 12.3 WordPiece — BERTテキスト テキスト

BPEテキスト テキスト **テキスト テキスト テキスト**:

$$\text{score}(a, b) = \frac{\text{freq}(ab)}{\text{freq}(a) \cdot \text{freq}(b)}$$

BPEテキスト テキスト度 テキスト テキスト, WordPieceテキスト テキスト度テキスト テキスト テキスト テキスト.
テキスト テキスト テキスト テキスト テキスト テキスト テキスト, テキスト テキスト テキスト テキスト テキスト.


In [ ]:
# WordPiece テキスト テキスト
class WordPiece:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []

    def _get_pair_scores(self, splits, word_freqs):
        pair_freqs = Counter()
        char_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
            for s in split:
                char_freqs[s] += word_freqs[word]
        scores = {p: f / (char_freqs[p[0]] * char_freqs[p[1]]) for p, f in pair_freqs.items()}
        return scores

    def _merge(self, pair, splits):
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        # WordPieceテキスト ## テキスト テキスト テキスト テキスト テキスト
        splits = {}
        for w in word_freqs:
            chars = list(w)
            splits[w] = [chars[0]] + ['##' + c for c in chars[1:]]
        for _ in range(self.num_merges):
            scores = self._get_pair_scores(splits, word_freqs)
            if not scores:
                break
            best = max(scores, key=scores.get)
            splits = self._merge(best, splits)
            self.merges.append(best)

wp = WordPiece(num_merges=10)
wp.train(corpus)
print("WordPiece Trainingテキスト テキストSum:")
for i, (a, b) in enumerate(wp.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")


## 12.4 Unigram Language Model — T5テキスト テキスト

BPE/WordPieceテキスト **bottom-up** (テキスト テキスト). Unigramテキスト **top-down** (テキスト テキスト テキスト).

1. テキスト テキスト = テキスト テキスト テキスト (テキスト)
2. テキスト テキスト テキスト $P(t) = \text{count}(t) / \sum \text{count}$
3. テキスト $\mathcal{L} = -\sum_i \log P(\mathbf{x}_i) = -\sum_i \log \sum_{\text{seg}} \prod P(t)$
4. テキスト テキスト テキスト テキスト テキスト テキスト
5. テキスト

EM テキスト テキスト テキスト テキスト.


In [ ]:
# Unigram テキスト テキスト (テキスト)
from itertools import combinations

class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}  # token -> prob

    def _get_candidates(self, word_freqs, max_len=10):
        candidates = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word)):
                for j in range(i+1, min(i+max_len+1, len(word)+1)):
                    candidates[word[i:j]] += freq
        return candidates

    def _segment(self, word, vocab):
        """Viterbiテキスト Optimal テキスト."""
        n = len(word)
        # dp[i] = (best_log_prob, best_segmentation)
        dp = [(-float('inf'), None)] * (n + 1)
        dp[0] = (0, [])
        for i in range(1, n + 1):
            for j in range(max(0, i - 10), i):
                token = word[j:i]
                if token in vocab:
                    log_p = np.log(vocab[token])
                    if dp[j][0] + log_p > dp[i][0]:
                        dp[i] = (dp[j][0] + log_p, dp[j][1] + [token])
        return dp[n]

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        candidates = self._get_candidates(word_freqs)
        total = sum(candidates.values())
        self.vocab = {t: c / total for t, c in candidates.items()}

        # Iterationテキスト Loss Increaseテキスト テキスト テキスト テキスト
        while len(self.vocab) > self.vocab_size:
            # テキスト テキスト テキスト テキスト Loss Increase Computation
            losses = {}
            for token in list(self.vocab.keys()):
                if len(token) == 1:  # テキスト テキスト テキスト
                    continue
                # テキスト テキスト テキスト Loss (テキスト)
                losses[token] = self.vocab[token]  # テキスト Probabilityテキスト lossテキスト
            if not losses:
                break
            # Lossテキスト テキスト テキスト テキスト テキスト
            sorted_tokens = sorted(losses.items(), key=lambda x: x[1])
            n_remove = max(1, len(self.vocab) - self.vocab_size)
            for token, _ in sorted_tokens[:n_remove]:
                del self.vocab[token]

# テキスト テキスト
uni = UnigramTokenizer(vocab_size=30)
uni.train(corpus)
print(f"Unigram Vocabulary Size: {len(uni.vocab)}")
print(f"テキスト テキスト: {sorted(uni.vocab.items(), key=lambda x: -x[1])[:10]}")

# テキスト テキスト
for word in ['lowest', 'newest']:
    log_p, seg = uni._segment(word, uni.vocab)
    print(f"  {word} -> {seg} (log_p={log_p:.4f})")


## 12.5 Byte-level BPE — GPT-2/3テキスト テキスト

問題: テキスト テキスト テキスト テキスト (テキスト 11,172テキスト, テキスト テキスト).
テキスト: テキスト テキスト **UTF-8 テキスト**テキスト エンコード テキスト BPE テキスト.

- テキスト 256テキスト → テキスト テキスト 256
- UNK(unknown) テキスト テキスト — テキスト テキスト テキスト テキスト
- テキスト テキスト テキスト

GPT-2 テキスト: 256 テキスト + 50,257 テキスト ≈ 50K テキスト


In [ ]:
# Byte-level BPE テキスト
text = "Hello, テキスト!"
bytes_seq = text.encode('utf-8')
print(f"Original: {text}")
print(f"UTF-8 テキスト: {list(bytes_seq)}")
print(f"テキスト テキスト: {len(bytes_seq)} (テキスト テキスト {len(text)})")
print("\n=> テキスト テキスト テキスト テキスト テキスト. UNK テキスト.")

# HuggingFace tokenizers テキスト テキスト
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
except ImportError:
    print("\n[HuggingFace tokenizers テキスト テキスト: pip install tokenizers]")

# テキスト BPE Tokenizer (HuggingFace)
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])

    # テキスト テキスト Training
    from llm_math.data import load_tiny_shakespeare
    train_text = load_tiny_shakespeare(max_chars=5000)
    tokenizer.train_from_iterator([train_text], trainer)

    print(f"\nTrainingテキスト Vocabulary Size: {tokenizer.get_vocab_size()}")
    # エンコード Test
    enc = tokenizer.encode("To be or not to be")
    print(f"'To be or not to be' -> {enc.tokens}")
    print(f"IDs: {enc.ids}")
except ImportError:
    print("\n[tokenizers テキスト テキスト. pip install tokenizersテキスト テキスト]")


## 12.6 [CPU/GPU ベンチマーク ④] トークナイザー 学習/エンコード 速度

トークナイザー テキスト CPUテキスト テキスト, テキスト テキスト テキスト テキスト 速度 テキスト テキスト.


In [ ]:
# トークナイザー ベンチマーク (CPUテキスト)
import time
from llm_math.data import load_tiny_shakespeare

# テキスト テキスト テキスト
text = load_tiny_shakespeare(max_chars=100000)
print(f"ベンチマークテキスト テキスト: {len(text):,}テキスト")

# 1. テキスト BPEテキスト 学習 時間 テキスト
def bench_bpe_train(text, n_merges, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        bpe = BPE(num_merges=n_merges)
        bpe.train([text])
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

print(f"\n{'merges':>8} | {'train (s)':>12}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    m, s = bench_bpe_train(text, n, repeat=2)
    print(f"{n:>8} | {m:>10.4f}±{s:.4f}")

# 2. エンコード Speed
bpe_small = BPE(num_merges=50)
bpe_small.train([text[:5000]])

def bench_encode(bpe, text, n_words=1000):
    words = text.split()[:n_words]
    t0 = time.perf_counter()
    for w in words:
        bpe.encode(w)
    return (time.perf_counter() - t0) * 1000

t_ms = bench_encode(bpe_small, text, n_words=1000)
print(f"\n1000テキスト エンコード: {t_ms:.2f}ms ({t_ms/1000:.3f}ms/テキスト)")
print("\n=> Tokenizerテキスト CPU テキスト. テキスト テキストPlane テキスト テキスト.")
print("   HuggingFace tokenizersテキスト Rust Implementationテキスト テキスト テキスト.")


## 12.7 テキスト テキスト 性能 テキスト

- テキスト テキスト → テキスト テキスト, モデル テキスト, テキスト テキスト テキスト
- テキスト テキスト → テキスト テキスト, テキスト テキスト テキスト, テキスト テキスト 学習 テキスト

LLMテキスト テキスト テキスト:
- GPT-2: 50,257
- GPT-3: 50,257
- LLaMA-2: 32,000
- LLaMA-3: 128,256
- GPT-4: ~100,000


In [ ]:
# テキスト テキスト シーケンス長 比較 (テキスト)
# テキスト テキスト テキスト テキスト テキスト テキスト
np.random.seed(0)
text_sample = load_tiny_shakespeare(max_chars=5000)

# テキスト テキスト テキスト 学習テキスト テキスト テキスト テキスト 比較
vocab_sizes = [30, 50, 100, 200, 500]
avg_tokens = []
for vs in vocab_sizes:
    bpe = BPE(num_merges=vs - 30)  # テキスト テキスト テキスト 30
    bpe.train([text_sample])
    # テキスト テキスト テキスト テキスト テキスト
    sample_words = text_sample.split()[:100]
    total_tokens = sum(len(bpe.encode(w)) for w in sample_words)
    avg_tokens.append(total_tokens / len(sample_words))

plt.figure(figsize=(9, 5))
plt.plot(vocab_sizes, avg_tokens, 'o-', linewidth=2, markersize=8)
plt.xlabel('Vocabulary Size')
plt.ylabel('Mean テキスト テキスト / word')
plt.title('Vocabulary Sizeテキスト Sequence Lengthテキスト Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch12_vocab_size_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> テキスト テキスト wordテキスト テキスト テキスト テキスト. テキスト Embedding テキスト テキスト.")


## 12.8 要点

| テキスト | テキスト テキスト | テキスト モデル |
|---|---|---|
| BPE | テキスト度 テキスト | GPT-2 |
| WordPiece | $\frac{\text{freq}(ab)}{\text{freq}(a)\text{freq}(b)}$ | BERT |
| Unigram | テキスト テキスト (top-down) | T5 |
| Byte-level BPE | BPE on bytes | GPT-2/3/4 |

## 演習問題

1. テキスト テキスト BPEテキスト テキスト テキスト テキスト, テキスト テキスト テキスト.
2. BPEテキスト WordPieceテキスト テキスト テキスト 学習テキスト テキスト テキスト 比較テキスト.
3. テキスト テキスト 100, 500, 1000テキスト テキスト テキスト テキスト テキスト 比較テキスト.
4. Byte-level BPEテキスト テキスト UNK テキスト テキスト テキスト.
5. HuggingFace `tokenizers` テキスト GPT-2 トークナイザーテキスト テキスト テキスト テキスト エンコードテキスト.

> 解答: `solutions/ch12_solutions.ipynb`
